In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qiskit Aer 프리미티브를 사용한 정확한 시뮬레이션 및 노이즈 시뮬레이션

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하시기를 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>
[Qiskit 프리미티브를 사용한 정확한 시뮬레이션](/guides/simulate-with-qiskit-sdk-primitives)에서는 Qiskit에 포함된 참조 프리미티브를 사용하여 양자 Circuit의 정확한 시뮬레이션을 수행하는 방법을 설명합니다. 현재 존재하는 양자 프로세서는 오류 또는 노이즈로 인해 어려움을 겪고 있으므로, 정확한 시뮬레이션의 결과가 실제 하드웨어에서 Circuit을 실행할 때 기대하는 결과와 반드시 일치하지는 않습니다. Qiskit의 참조 프리미티브는 노이즈 모델링을 지원하지 않지만, [Qiskit Aer](https://qiskit.org/ecosystem/aer/)에는 노이즈 모델링을 지원하는 프리미티브 구현이 포함되어 있습니다. Qiskit Aer는 고성능 양자 Circuit 시뮬레이터로, 더 나은 성능과 더 많은 기능을 위해 참조 프리미티브 대신 사용할 수 있습니다. Qiskit Aer는 [Qiskit Ecosystem](https://qiskit.github.io/ecosystem/)의 일부입니다. 이 문서에서는 정확한 시뮬레이션 및 노이즈 시뮬레이션을 위한 Qiskit Aer 프리미티브의 사용법을 설명합니다.

> **Note:** - `qiskit-aer` v0.14 이상이 필요합니다.
> - Qiskit Aer 프리미티브는 프리미티브 인터페이스를 구현하지만, Qiskit Runtime 프리미티브와 동일한 옵션을 제공하지는 않습니다. 예를 들어, 복원력 수준(Resilience level)은 Qiskit Aer 프리미티브에서 사용할 수 없습니다.
> - Aer가 지원하는 시뮬레이션 방법 옵션에 대한 자세한 내용은 [AerSimulator 문서](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator)를 참조하세요.

정확한 시뮬레이션과 노이즈 시뮬레이션을 탐색하기 위해 8개의 Qubit으로 예제 Circuit을 생성합니다.

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg)

이 Circuit에는 $R_y$ 및 $R_z$ Gate의 회전 각도를 나타내는 매개변수가 포함되어 있습니다. 이 Circuit을 시뮬레이션할 때는 이러한 매개변수에 대한 명시적인 값을 지정해야 합니다. 다음 셀에서는 이러한 매개변수에 대한 일부 값을 지정하고, Qiskit Aer의 Estimator 프리미티브를 사용하여 관측량 $ZZ \cdots Z$의 정확한 기대값을 계산합니다.

In [2]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)
params = [0.1] * circuit.num_parameters

exact_estimator = Estimator()
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.8870140234256602

Now, let's initialize a noise model that includes depolarizing error of 2% on every CX gate. In practice, the error arising from the two-qubit gates, which are CX gates here, are the dominant source of error when running a circuit. See [Build noise models](./build-noise-models) for an overview of constructing noise models in Qiskit Aer.

In the next cell, we construct an Estimator that incorporates this noise model and use it to compute the expectation value of the observable.

In [3]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)

noisy_estimator = Estimator(
    options=dict(backend_options=dict(noise_model=noise_model))
)
job = noisy_estimator.run([pub])
result = job.result()
pub_result = result[0]
noisy_value = float(pub_result.data.evs)
noisy_value

0.7247404214143528

이제 모든 CX Gate에 2%의 탈분극 오류(depolarizing error)를 포함하는 노이즈 모델을 초기화해 보겠습니다. 실제로는 여기서 CX Gate인 2큐비트 Gate에서 발생하는 오류가 Circuit을 실행할 때 오류의 주요 원인입니다. Qiskit Aer에서 노이즈 모델 구성에 대한 개요는 [노이즈 모델 구축](./build-noise-models)을 참조하세요.

다음 셀에서는 이 노이즈 모델을 포함하는 Estimator를 구성하고 이를 사용하여 관측량의 기대값을 계산합니다.

In [4]:
cx_count = circuit.count_ops()["cx"]
(1 - cx_depolarizing_prob) ** cx_count

0.6542558123199923

This value, 65%, gives a rough estimate of the probability that our final state is correct. It is a conservative estimate because it does not take into account the initial state of the simulation.

The following code cell shows how to use the Sampler primitive from Qiskit Aer to sample from the noisy circuit. We need to add measurements to the circuit before running it with the Sampler primitive.

In [5]:
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

noisy_sampler = Sampler(
    options=dict(backend_options=dict(noise_model=noise_model))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params, 100)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
pub_result.data.meas.get_counts()

{'00100000': 1,
 '00000000': 65,
 '10101000': 1,
 '10000000': 5,
 '00001000': 1,
 '00000110': 2,
 '11110010': 1,
 '00000011': 3,
 '01010000': 3,
 '11000000': 3,
 '01111000': 1,
 '01000000': 2,
 '00000010': 1,
 '01100000': 1,
 '00011000': 1,
 '00111100': 1,
 '00010100': 1,
 '00001111': 1,
 '00110000': 1,
 '01100101': 1,
 '00000100': 1,
 '10100000': 1,
 '00000001': 1,
 '11010000': 1}

보시다시피, 노이즈가 있을 때의 기대값은 올바른 값과 상당히 차이가 납니다. 실제로는 노이즈의 영향을 상쇄하기 위해 다양한 오류 완화 기술을 사용할 수 있지만, 이러한 기술에 대한 논의는 이 문서의 범위를 벗어납니다.

노이즈가 최종 결과에 어떻게 영향을 미치는지 대략적으로 이해하기 위해, 각 CX Gate에 2%의 탈분극 오류를 추가하는 노이즈 모델을 고려해 보겠습니다. 확률 $p$의 탈분극 오류는 밀도 행렬 $\rho$에 대해 다음과 같은 작용을 하는 양자 채널 $E$로 정의됩니다.

$$
E(\rho) = (1 - p) \rho + p\frac{I}{2^n}
$$

여기서 $n$은 Qubit의 수로, 이 경우 2입니다. 즉, 확률 $p$로 상태가 완전히 혼합된 상태로 대체되고, 확률 $1 - p$로 상태가 보존됩니다. 탈분극 채널의 $m$번 적용 후 상태가 보존될 확률은 $(1 - p)^m$이 됩니다. 따라서 우리 Circuit의 CX Gate 수에 따라 시뮬레이션 끝에 올바른 상태를 유지할 확률이 기하급수적으로 감소할 것으로 예상됩니다.

Circuit에서 CX Gate의 수를 세고 $(1 - p)^m$을 계산해 보겠습니다. `count_ops`를 호출하여 Gate 이름을 횟수에 매핑하는 딕셔너리를 얻고 CX Gate에 대한 항목을 검색합니다.